# Table 2 joint baseline system

The joint conditional Gaussian likelihood uses independent equation errors to identify the recursive return equation. The return cell checks its unrestricted estimates against native OLS reparameterizations and estimates both nested one-parameter efficiency tests. The accompanying `design.md` documents currency, benchmark, and source-method choices.


In [1]:
version 18
set more off

use "data/order_backlog_firm_years/reg.dta", clear
est clear

count if missing(earn_lead, sar, earn_t, bklg)
if r(N) != 0 {
    display as error "The declared common sample contains missing system inputs."
    exit 459
}
count
local commonN = r(N)
display as text "Common Table 2 sample N = " `commonN'

mlexp (lnnormalden(earn_lead, {omega0=0} + {omega1=0}*earn_t + {omega2=0}*bklg, exp({lnsige=0})) ///
       + lnnormalden(sar, {beta0=0} + {beta1=1}*(earn_lead - {omega0} - {omega1star=0}*earn_t - {omega2star=0}*bklg), ///
                     exp({lnsigsar=0}))), variables(earn_lead sar earn_t bklg)
if e(converged) != 1 {
    display as error "Unrestricted independent-error Gaussian likelihood did not converge."
    exit 498
}
estimates store unrestricted
local fitN = e(N)
if `fitN' != `commonN' {
    display as error "Unrestricted likelihood changed the common sample."
    exit 459
}
scalar ll_unrestricted = e(ll)

mlexp (lnnormalden(earn_lead, {omega0=0} + {omega1=0}*earn_t + {omega2=0}*bklg, exp({lnsige=0})) ///
       + lnnormalden(sar, {beta0=0} + {beta1=1}*(earn_lead - {omega0} - {omega1}*earn_t - {omega2star=0}*bklg), ///
                     exp({lnsigsar=0}))), variables(earn_lead sar earn_t bklg)
if e(converged) != 1 {
    display as error "The omega1 efficiency restriction did not converge."
    exit 498
}
estimates store constrained_omega1
if e(N) != `commonN' {
    display as error "The omega1 restriction changed the common sample."
    exit 459
}
lrtest unrestricted constrained_omega1, df(1)
scalar LR_omega1 = r(chi2)
scalar p_omega1 = r(p)

mlexp (lnnormalden(earn_lead, {omega0=0} + {omega1=0}*earn_t + {omega2=0}*bklg, exp({lnsige=0})) ///
       + lnnormalden(sar, {beta0=0} + {beta1=1}*(earn_lead - {omega0} - {omega1star=0}*earn_t - {omega2}*bklg), ///
                     exp({lnsigsar=0}))), variables(earn_lead sar earn_t bklg)
if e(converged) != 1 {
    display as error "The omega2 efficiency restriction did not converge."
    exit 498
}
estimates store constrained_omega2
if e(N) != `commonN' {
    display as error "The omega2 restriction changed the common sample."
    exit 459
}
lrtest unrestricted constrained_omega2, df(1)
scalar LR_omega2 = r(chi2)
scalar p_omega2 = r(p)

display as text "Independent-error conditional Gaussian likelihoods: unrestricted=" %12.3f scalar(ll_unrestricted)

display as text "Native likelihood-ratio restrictions each have one degree of freedom."

regress earn_lead earn_t bklg
scalar ols_omega0 = _b[_cons]
scalar ols_omega1 = _b[earn_t]
scalar ols_omega2 = _b[bklg]
scalar ols_sigma_forecast = sqrt(e(rss)/e(N))

regress sar earn_lead earn_t bklg
scalar ols_reduced_intercept = _b[_cons]
scalar ols_beta1 = _b[earn_lead]
if abs(scalar(ols_beta1)) < 1e-10 {
    display as error "Conditional return coefficient is zero; structural reparameterization is undefined."
    exit 459
}
scalar ols_omega1star = -_b[earn_t]/_b[earn_lead]
scalar ols_omega2star = -_b[bklg]/_b[earn_lead]
scalar ols_beta0 = _b[_cons] + _b[earn_lead]*scalar(ols_omega0)
scalar ols_sigma_return = sqrt(e(rss)/e(N))

estimates restore unrestricted
scalar gap_omega0 = _b[/omega0] - scalar(ols_omega0)
scalar gap_omega1 = _b[/omega1] - scalar(ols_omega1)
scalar gap_omega2 = _b[/omega2] - scalar(ols_omega2)
scalar gap_beta0 = _b[/beta0] - scalar(ols_beta0)
scalar gap_beta1 = _b[/beta1] - scalar(ols_beta1)
scalar gap_omega1star = _b[/omega1star] - scalar(ols_omega1star)
scalar gap_omega2star = _b[/omega2star] - scalar(ols_omega2star)
scalar gap_sigma_forecast = exp(_b[/lnsige]) - scalar(ols_sigma_forecast)
scalar gap_sigma_return = exp(_b[/lnsigsar]) - scalar(ols_sigma_return)
scalar gap_abs = max(abs(scalar(gap_omega0)), abs(scalar(gap_omega1)), abs(scalar(gap_omega2)), ///
    abs(scalar(gap_beta0)), abs(scalar(gap_beta1)), abs(scalar(gap_omega1star)), ///
    abs(scalar(gap_omega2star)), abs(scalar(gap_sigma_forecast)), abs(scalar(gap_sigma_return)))
scalar gap_rel = max(abs(scalar(gap_omega0))/(1+abs(scalar(ols_omega0))), ///
    abs(scalar(gap_omega1))/(1+abs(scalar(ols_omega1))), ///
    abs(scalar(gap_omega2))/(1+abs(scalar(ols_omega2))), ///
    abs(scalar(gap_beta0))/(1+abs(scalar(ols_beta0))), ///
    abs(scalar(gap_beta1))/(1+abs(scalar(ols_beta1))), ///
    abs(scalar(gap_omega1star))/(1+abs(scalar(ols_omega1star))), ///
    abs(scalar(gap_omega2star))/(1+abs(scalar(ols_omega2star))), ///
    abs(scalar(gap_sigma_forecast))/(1+abs(scalar(ols_sigma_forecast))), ///
    abs(scalar(gap_sigma_return))/(1+abs(scalar(ols_sigma_return))))
display as text "Unrestricted structural-to-OLS mapping: max absolute gap = " %12.8g scalar(gap_abs) ///
    "; max scale-relative gap = " %12.8g scalar(gap_rel)
display as text "Estimated positive error SDs: forecast=" %10.6g exp(_b[/lnsige]) ///
    " return=" %10.6g exp(_b[/lnsigsar])

estadd scalar LR_omega1 = scalar(LR_omega1)
estadd scalar p_omega1 = scalar(p_omega1)
estadd scalar LR_omega2 = scalar(LR_omega2)
estadd scalar p_omega2 = scalar(p_omega2)
estadd scalar sigma_forecast = exp(_b[/lnsige])
estadd scalar sigma_return = exp(_b[/lnsigsar])
est store m_table2

esttab m_table2 using "tables/table2_forecasting_equation.md", ///
    cells(b(star fmt(3)) p(par)) starlevels(* 0.1 ** 0.05 *** 0.01) ///
    keep(omega0:_cons omega1:_cons omega2:_cons) ///
    coeflabels(omega0:_cons "omega0 (shared intercept)" omega1:_cons "omega1" omega2:_cons "omega2 (order backlog)") ///
    stats(N ll LR_omega1 p_omega1 LR_omega2 p_omega2 sigma_forecast sigma_return, ///
    labels("N" "Log likelihood" "LR: omega1 = omega1*" "p-value" ///
           "LR: omega2 = omega2*" "p-value" "Forecast error SD" "Return error SD") ///
    fmt(0 3 2 4 2 4 3 3)) eqlabels(none) replace label
esttab m_table2 using "tables/table2_forecasting_equation.rtf", ///
    cells(b(star fmt(3)) p(par)) starlevels(* 0.1 ** 0.05 *** 0.01) ///
    keep(omega0:_cons omega1:_cons omega2:_cons) ///
    coeflabels(omega0:_cons "omega0 (shared intercept)" omega1:_cons "omega1" omega2:_cons "omega2 (order backlog)") ///
    stats(N ll LR_omega1 p_omega1 LR_omega2 p_omega2 sigma_forecast sigma_return, ///
    labels("N" "Log likelihood" "LR: omega1 = omega1*" "p-value" ///
           "LR: omega2 = omega2*" "p-value" "Forecast error SD" "Return error SD") ///
    fmt(0 3 2 4 2 4 3 3)) eqlabels(none) replace label
esttab m_table2 using "tables/table2_forecasting_equation.csv", ///
    cells(b(star fmt(3)) p(par)) starlevels(* 0.1 ** 0.05 *** 0.01) ///
    keep(omega0:_cons omega1:_cons omega2:_cons) ///
    coeflabels(omega0:_cons "omega0 (shared intercept)" omega1:_cons "omega1" omega2:_cons "omega2 (order backlog)") ///
    stats(N ll LR_omega1 p_omega1 LR_omega2 p_omega2 sigma_forecast sigma_return, ///
    labels("N" "Log likelihood" "LR: omega1 = omega1*" "p-value" ///
           "LR: omega2 = omega2*" "p-value" "Forecast error SD" "Return error SD") ///
    fmt(0 3 2 4 2 4 3 3)) eqlabels(none) replace label
esttab m_table2 using "tables/table2_forecasting_equation.tex", ///
    cells(b(star fmt(3)) p(par)) starlevels(* 0.1 ** 0.05 *** 0.01) ///
    keep(omega0:_cons omega1:_cons omega2:_cons) ///
    coeflabels(omega0:_cons "omega0 (shared intercept)" omega1:_cons "omega1" omega2:_cons "omega2 (order backlog)") ///
    stats(N ll LR_omega1 p_omega1 LR_omega2 p_omega2 sigma_forecast sigma_return, ///
    labels("N" "Log likelihood" "LR: omega1 = omega1*" "p-value" ///
           "LR: omega2 = omega2*" "p-value" "Forecast error SD" "Return error SD") ///
    fmt(0 3 2 4 2 4 3 3)) eqlabels(none) replace label

esttab m_table2 using "tables/table2_returns_equation.md", ///
    cells(b(star fmt(3)) p(par)) starlevels(* 0.1 ** 0.05 *** 0.01) ///
    keep(omega0:_cons omega1star:_cons omega2star:_cons beta0:_cons beta1:_cons) ///
    coeflabels(omega0:_cons "omega0 (shared intercept)" omega1star:_cons "omega1*" ///
               omega2star:_cons "omega2* (order backlog)" beta0:_cons "beta0" beta1:_cons "beta1") ///
    stats(N ll LR_omega1 p_omega1 LR_omega2 p_omega2 sigma_forecast sigma_return, ///
    labels("N" "Log likelihood" "LR: omega1 = omega1*" "p-value" ///
           "LR: omega2 = omega2*" "p-value" "Forecast error SD" "Return error SD") ///
    fmt(0 3 2 4 2 4 3 3)) eqlabels(none) replace label
esttab m_table2 using "tables/table2_returns_equation.rtf", ///
    cells(b(star fmt(3)) p(par)) starlevels(* 0.1 ** 0.05 *** 0.01) ///
    keep(omega0:_cons omega1star:_cons omega2star:_cons beta0:_cons beta1:_cons) ///
    coeflabels(omega0:_cons "omega0 (shared intercept)" omega1star:_cons "omega1*" ///
               omega2star:_cons "omega2* (order backlog)" beta0:_cons "beta0" beta1:_cons "beta1") ///
    stats(N ll LR_omega1 p_omega1 LR_omega2 p_omega2 sigma_forecast sigma_return, ///
    labels("N" "Log likelihood" "LR: omega1 = omega1*" "p-value" ///
           "LR: omega2 = omega2*" "p-value" "Forecast error SD" "Return error SD") ///
    fmt(0 3 2 4 2 4 3 3)) eqlabels(none) replace label
esttab m_table2 using "tables/table2_returns_equation.csv", ///
    cells(b(star fmt(3)) p(par)) starlevels(* 0.1 ** 0.05 *** 0.01) ///
    keep(omega0:_cons omega1star:_cons omega2star:_cons beta0:_cons beta1:_cons) ///
    coeflabels(omega0:_cons "omega0 (shared intercept)" omega1star:_cons "omega1*" ///
               omega2star:_cons "omega2* (order backlog)" beta0:_cons "beta0" beta1:_cons "beta1") ///
    stats(N ll LR_omega1 p_omega1 LR_omega2 p_omega2 sigma_forecast sigma_return, ///
    labels("N" "Log likelihood" "LR: omega1 = omega1*" "p-value" ///
           "LR: omega2 = omega2*" "p-value" "Forecast error SD" "Return error SD") ///
    fmt(0 3 2 4 2 4 3 3)) eqlabels(none) replace label
esttab m_table2 using "tables/table2_returns_equation.tex", ///
    cells(b(star fmt(3)) p(par)) starlevels(* 0.1 ** 0.05 *** 0.01) ///
    keep(omega0:_cons omega1star:_cons omega2star:_cons beta0:_cons beta1:_cons) ///
    coeflabels(omega0:_cons "omega0 (shared intercept)" omega1star:_cons "omega1*" ///
               omega2star:_cons "omega2* (order backlog)" beta0:_cons "beta0" beta1:_cons "beta1") ///
    stats(N ll LR_omega1 p_omega1 LR_omega2 p_omega2 sigma_forecast sigma_return, ///
    labels("N" "Log likelihood" "LR: omega1 = omega1*" "p-value" ///
           "LR: omega2 = omega2*" "p-value" "Forecast error SD" "Return error SD") ///
    fmt(0 3 2 4 2 4 3 3)) eqlabels(none) replace label
est clear



  ___  ____  ____  ____  ____ ®
 /__    /   ____/   /   ____/      Stata 18.0
___/   /   /___/   /   /___/       MP—Parallel Edition

 Statistics and Data Science       Copyright 1985-2023 StataCorp LLC
                                   StataCorp
                                   4905 Lakeway Drive
                                   College Station, Texas 77845 USA
                                   800-782-8272        https://www.stata.com
                                   979-696-4600        service@stata.com

Stata license: Unlimited-user 16-core network, expiring 15 Dec 2024
Serial number: 501809303220
  Licensed to: Luther
               Users

Notes:
      1. Stata is running in batch mode.
      2. Unicode is supported; see help unicode_advice.
      3. More than 2 billion observations are allowed; see help obs_advice.
      4. Maximum number of variables is set to 5,000 but can be increased;
          see help set_maxvar.


Running /usr/local/stata18/sysprofile.do ...

. do

## Descriptive statistics

These statistics use the realized common Table 2 regression sample.


In [1]:
version 18
set more off

use "data/order_backlog_firm_years/reg.dta", clear
est clear

estpost tabstat earn_t earn_lead bklg sar, statistics(n mean sd p25 p50 p75) columns(statistics)
esttab using "tables/table2_descriptives.md", ///
    cells("count(fmt(0)) mean(fmt(3)) sd(fmt(3)) p25(fmt(3)) p50(fmt(3)) p75(fmt(3))") ///
    replace label
esttab using "tables/table2_descriptives.rtf", ///
    cells("count(fmt(0)) mean(fmt(3)) sd(fmt(3)) p25(fmt(3)) p50(fmt(3)) p75(fmt(3))") ///
    replace label
esttab using "tables/table2_descriptives.csv", ///
    cells("count(fmt(0)) mean(fmt(3)) sd(fmt(3)) p25(fmt(3)) p50(fmt(3)) p75(fmt(3))") ///
    replace label
esttab using "tables/table2_descriptives.tex", ///
    cells("count(fmt(0)) mean(fmt(3)) sd(fmt(3)) p25(fmt(3)) p50(fmt(3)) p75(fmt(3))") ///
    replace label
est clear



  ___  ____  ____  ____  ____ ®
 /__    /   ____/   /   ____/      Stata 18.0
___/   /   /___/   /   /___/       MP—Parallel Edition

 Statistics and Data Science       Copyright 1985-2023 StataCorp LLC
                                   StataCorp
                                   4905 Lakeway Drive
                                   College Station, Texas 77845 USA
                                   800-782-8272        https://www.stata.com
                                   979-696-4600        service@stata.com

Stata license: Unlimited-user 16-core network, expiring 15 Dec 2024
Serial number: 501809303220
  Licensed to: Luther
               Users

Notes:
      1. Stata is running in batch mode.
      2. Unicode is supported; see help unicode_advice.
      3. More than 2 billion observations are allowed; see help obs_advice.
      4. Maximum number of variables is set to 5,000 but can be increased;
          see help set_maxvar.


Running /usr/local/stata18/sysprofile.do ...

. do